# Flow Matching for Protein Design
## Continuous Normalizing Flows · ODE Solvers · FrameDiff · Boltz-1 · SE(3) Manifolds

---

## TL;DR

**Plain English:** Flow matching is a way to train a generative model by teaching a neural network to push particles along straight (or nearly straight) paths from a simple source distribution (like Gaussian noise) to a complex target distribution (like protein structures). Instead of the thousands of noisy steps diffusion needs, flow matching can travel the same journey in far fewer steps because the paths are smoother.

**Why it matters for structural biology:** AlphaFold3, Boltz-1, and FrameDiff all use variants of flow matching (or score/flow hybrids) to generate 3D protein structures. Understanding this framework lets you read the latest structure prediction and protein design papers fluently.

**What you will be able to do after this notebook:**
- Derive the conditional flow matching (CFM) objective from first principles
- Implement a vector field network and an Euler-step ODE solver from scratch
- Train a flow matching model on a 2D toy example and watch it converge
- Explain how FrameDiff lifts flow matching to the SE(3) manifold
- Articulate why Boltz-1 replaced DDPM with flow matching and what the trade-offs are
- Answer graduate-level interview questions comparing flow matching and diffusion

| | |
|---|---|
| **Estimated time** | 3–4 hours (first pass); 1 hour (concept-only pass) |
| **Difficulty** | Advanced undergraduate / early graduate |
| **Prerequisites** | Module 12/01 (DDPM), Module 07 (AF3 diffusion head), calculus (ODEs), basic PyTorch |
| **Key papers** | Lipman et al. 2022 (arxiv:2210.02747), Yim et al. 2023 (arxiv:2302.02277), Abramson et al. 2024 (Nature) |

## 1. Beginner Frame — What Is Flow Matching?

### The Core Idea in Plain English

Imagine you have a bucket of sand (noise) and you want it to look like a sand sculpture (a protein). Diffusion models do this by:
1. Crumbling a real sculpture grain by grain (forward process — adding noise over 1000 steps)
2. Learning to reverse each crumbling step (reverse process — denoising)

Flow matching does something cleaner:
1. For each grain of sand, draw a **straight-line path** from where it starts (in noise) to where it needs to end up (in the sculpture)
2. Train a network to predict, at any point in time, what **direction** each grain should be moving
3. At inference time, follow those directions from noise to structure

The key insight: you never need to simulate the full path during training. You only need to **match the instantaneous velocity** at randomly sampled time points. This makes training much more efficient.

### Vocabulary Table

| Term | Plain English | Math Symbol |
|------|--------------|-------------|
| **Source distribution** | Where we start — usually Gaussian noise | $p_0 = \mathcal{N}(0, I)$ |
| **Target distribution** | Where we want to end up — real data (protein structures) | $p_1 = p_{\text{data}}$ |
| **Probability path** | A smooth interpolation of distributions from $p_0$ to $p_1$ over time $t \in [0,1]$ | $p_t(x)$ |
| **Vector field** | At every point in space-time, the direction a particle should move | $u_t(x)$ |
| **Conditional vector field** | The ideal direction for a particle given it will end at a specific data point $x_1$ | $u_t(x \mid x_1)$ |
| **ODE** | Ordinary Differential Equation — describes how particle positions evolve over time | $\frac{dx}{dt} = u_t(x_t)$ |
| **Euler step** | Simplest ODE solver: move a tiny bit in the current direction | $x_{t+h} = x_t + h \cdot u_t(x_t)$ |
| **Continuous normalizing flow (CNF)** | A generative model defined by an ODE; changes of variables are tracked via the flow | — |
| **Score function** | Gradient of the log-density; used in diffusion but not flow matching directly | $\nabla_x \log p_t(x)$ |
| **SE(3)** | The group of rigid-body motions in 3D: rotations + translations; protein frames live here | $R \in SO(3), t \in \mathbb{R}^3$ |
| **Marginal vector field** | The average conditional vector field over all possible endpoints | $u_t(x) = \mathbb{E}[u_t(x\mid x_1)\mid x_t=x]$ |

## 2. Mathematical Foundations

### 2.1 Continuous Normalizing Flows (CNFs)

A continuous normalizing flow defines a time-dependent map $\phi_t : \mathbb{R}^d \to \mathbb{R}^d$ via an ODE:

$$\frac{d}{dt}\phi_t(x) = u_t(\phi_t(x)), \quad \phi_0(x) = x$$

If $X_0 \sim p_0$, then $X_t = \phi_t(X_0)$ follows distribution $p_t$. The density evolves according to the **continuity equation**:

$$\frac{\partial p_t}{\partial t} + \nabla \cdot (p_t \, u_t) = 0$$

This is the fundamental PDE linking probability paths to vector fields. If we know the probability path $p_t$, we can (in principle) recover the vector field $u_t$ that generates it.

### 2.2 The Flow Matching Objective

The **flow matching** (FM) loss (Lipman et al. 2022) trains a neural network $v_\theta$ to match the marginal vector field:

$$\mathcal{L}_{\text{FM}}(\theta) = \mathbb{E}_{t, p_t(x)} \left[ \| v_\theta(x, t) - u_t(x) \|^2 \right]$$

**Problem:** We cannot evaluate $u_t(x)$ directly — it requires integrating over all data points.

**Solution — Conditional Flow Matching (CFM):** Condition on a specific endpoint $x_1 \sim p_{\text{data}}$. Define:
- A **conditional probability path** $p_t(x \mid x_1)$ — the distribution of particles at time $t$ given they will end at $x_1$
- A **conditional vector field** $u_t(x \mid x_1)$ — the velocity of a particle on the optimal path from $x_0$ to $x_1$

The **CFM loss** is:

$$\mathcal{L}_{\text{CFM}}(\theta) = \mathbb{E}_{t, q(x_1), p_t(x \mid x_1)} \left[ \| v_\theta(x, t) - u_t(x \mid x_1) \|^2 \right]$$

**Key theorem (Lipman et al. 2022):** $\mathcal{L}_{\text{FM}}$ and $\mathcal{L}_{\text{CFM}}$ have the same gradient with respect to $\theta$. Therefore, minimising the tractable CFM objective is equivalent to minimising the intractable FM objective.

### 2.3 The Optimal Transport Probability Path

The simplest and most powerful choice: **linear interpolation** between $x_0 \sim p_0$ and $x_1 \sim p_{\text{data}}$:

$$x_t = (1 - t) x_0 + t x_1, \quad t \in [0, 1]$$

This gives the conditional vector field:

$$u_t(x \mid x_1) = \frac{d x_t}{dt} = x_1 - x_0$$

**This is a constant!** The target velocity is simply the displacement from source to target, independent of $t$. This is why flow matching with OT paths is so clean to implement: the network just needs to predict $x_1 - x_0$.

### 2.4 Training Algorithm

```
For each training step:
  1. Sample x_1 ~ p_data           (a real data point)
  2. Sample x_0 ~ N(0, I)          (noise)
  3. Sample t ~ Uniform[0, 1]      (random time)
  4. Compute x_t = (1-t)*x_0 + t*x_1  (interpolated point)
  5. Target velocity: u = x_1 - x_0
  6. Predicted velocity: v = network(x_t, t)
  7. Loss = ||v - u||^2
  8. Backprop and update
```

### 2.5 Sampling (Inference)

To generate a new sample:
1. Draw $x_0 \sim \mathcal{N}(0, I)$
2. Integrate the ODE $\frac{dx}{dt} = v_\theta(x_t, t)$ from $t=0$ to $t=1$ using an ODE solver
3. $x_1$ is your generated sample

Because the paths are nearly straight (especially with OT), you can get good samples with **very few function evaluations** (NFE). DDPM typically needs 100–1000 steps; flow matching can work in 10–50.

## 3. Flow Matching vs DDPM: Side-by-Side

| Aspect | DDPM (Diffusion) | Flow Matching (OT-CFM) |
|--------|-----------------|------------------------|
| **Forward process** | $q(x_t \mid x_0) = \mathcal{N}(\sqrt{\bar\alpha_t} x_0, (1-\bar\alpha_t)I)$ — multiplicative noise schedule | $x_t = (1-t)x_0 + tx_1$ — linear interpolation |
| **Network target** | Predict noise $\epsilon$ added at step $t$ (or score $\nabla \log p_t$) | Predict vector field $u_t = x_1 - x_0$ (velocity) |
| **Training objective** | $\mathbb{E}[\|\epsilon_\theta(x_t,t) - \epsilon\|^2]$ | $\mathbb{E}[\|v_\theta(x_t,t) - (x_1 - x_0)\|^2]$ |
| **Sampling** | Iterative DDPM/DDIM reverse steps (100–1000 NFE) | ODE integration (10–50 NFE typical) |
| **Path geometry** | Curved, non-straight trajectories | Straight-line (OT) trajectories |
| **Theoretical basis** | SDEs, score matching | ODEs, optimal transport |
| **Noise schedule** | Carefully tuned ($\beta_t$ schedule critical) | None needed for basic OT-CFM |
| **Likelihood** | Lower bound via ELBO | Exact via change-of-variables (CNF) |
| **Protein design use** | RFdiffusion, early AF3 drafts | FrameDiff (SE3-FM), Boltz-1, AF3 final |
| **Key advantage** | Well-understood, stable training | Fewer sampling steps, cleaner math |
| **Key weakness** | Slow sampling, curved paths hard to generalise | Newer, fewer off-the-shelf implementations |

### Connection: Both Are the Same SDE at Infinity

Flow matching and diffusion are not unrelated. As shown in Song et al. (2021), diffusion models can be viewed as SDEs with a specific drift and diffusion coefficient. Flow matching with straight paths corresponds to the **probability flow ODE** — the deterministic counterpart of the diffusion SDE. The key difference is that flow matching removes the stochastic noise term entirely, giving smoother paths.

## 4. Implementation: Flow Matching from Scratch (1D)

We start with the simplest possible case: flow matching in 1D, mapping $\mathcal{N}(0,1)$ to a bimodal distribution.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

> **Expected output:** Device information (e.g., `Using device: cpu` or `Using device: cuda`), confirming PyTorch is loaded and the compute device is detected  
> If you see this, your code is working correctly.  
> If you see an error, check the Troubleshooting section or re-run the cell.

In [ ]:
# ── 4.1  Target distribution: 1D bimodal Gaussian ──────────────────────────

def sample_bimodal(n: int) -> torch.Tensor:
    """Sample from 0.5*N(-2, 0.5) + 0.5*N(2, 0.5)."""
    half = n // 2
    left  = torch.randn(half, 1) * 0.5 - 2.0
    right = torch.randn(n - half, 1) * 0.5 + 2.0
    samples = torch.cat([left, right], dim=0)
    # shuffle so we don't feed sorted batches
    idx = torch.randperm(n)
    return samples[idx]

# Quick visual
data = sample_bimodal(5000).numpy()
noise = np.random.randn(5000, 1)

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].hist(noise, bins=60, density=True, color='steelblue', alpha=0.7)
axes[0].set_title('Source: $\\mathcal{N}(0,1)$')
axes[0].set_xlabel('x')
axes[1].hist(data, bins=60, density=True, color='coral', alpha=0.7)
axes[1].set_title('Target: Bimodal Gaussian')
axes[1].set_xlabel('x')
plt.suptitle('1D Flow Matching: Source and Target Distributions', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f'Source mean={noise.mean():.3f}, std={noise.std():.3f}')
print(f'Target mean={data.mean():.3f}, std={data.std():.3f}')

In [ ]:
# ── 4.2  Vector Field Network ───────────────────────────────────────────────

class VectorFieldNet1D(nn.Module):
    """
    Predicts the vector field u_t(x) for 1D flow matching.

    Input:  (x_t, t)  — concatenated as a 2D vector
    Output: v_hat     — predicted velocity (scalar in 1D)

    Architecture: MLP with sinusoidal time embedding + residual connections.
    The sinusoidal embedding is critical: it helps the network distinguish
    nearby time values (e.g. t=0.49 vs t=0.51) which look identical to a
    raw scalar encoding.
    """

    def __init__(self, hidden_dim: int = 128, n_layers: int = 4,
                 time_embed_dim: int = 32):
        super().__init__()
        self.time_embed_dim = time_embed_dim

        # Sinusoidal time embedding (fixed frequencies, learned projection)
        freqs = torch.exp(
            torch.arange(0, time_embed_dim, 2).float() *
            -(np.log(10000.0) / time_embed_dim)
        )
        self.register_buffer('freqs', freqs)  # shape: (time_embed_dim/2,)

        # Project [sin, cos] embeddings to hidden_dim
        self.time_proj = nn.Sequential(
            nn.Linear(time_embed_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

        # Input projection: x_t is 1D
        self.x_proj = nn.Linear(1, hidden_dim)

        # Residual MLP blocks
        layers = []
        for _ in range(n_layers):
            layers.append(nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.SiLU(),
                nn.Linear(hidden_dim, hidden_dim),
            ))
        self.res_blocks = nn.ModuleList(layers)

        # Output head
        self.out = nn.Linear(hidden_dim, 1)

    def time_embedding(self, t: torch.Tensor) -> torch.Tensor:
        """t: (B,)  ->  embedding: (B, time_embed_dim)"""
        t = t.unsqueeze(-1)   # (B, 1)
        args = t * self.freqs  # (B, time_embed_dim/2)
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
        return emb  # (B, time_embed_dim)

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """
        x: (B, 1) — current position
        t: (B,)   — time in [0, 1]
        returns: (B, 1) — predicted velocity
        """
        h = self.x_proj(x)                       # (B, hidden_dim)
        h = h + self.time_proj(self.time_embedding(t))  # add time info

        for block in self.res_blocks:
            h = h + block(h)  # residual connection

        return self.out(h)   # (B, 1)


# Sanity check: forward pass
net = VectorFieldNet1D().to(device)
dummy_x = torch.randn(16, 1).to(device)
dummy_t = torch.rand(16).to(device)
v_out = net(dummy_x, dummy_t)
print(f'Network output shape: {v_out.shape}  (expected: [16, 1])')
print(f'Total parameters: {sum(p.numel() for p in net.parameters()):,}')

In [ ]:
# ── 4.3  Training Loop ──────────────────────────────────────────────────────

def train_flow_matching_1d(
    net: nn.Module,
    n_steps: int = 5000,
    batch_size: int = 512,
    lr: float = 3e-4,
    log_every: int = 500,
) -> list:
    """
    Conditional Flow Matching training with OT paths.

    At each step:
      x_1  ~ p_data  (real data)
      x_0  ~ N(0,I)  (source noise)
      t    ~ U[0,1]  (random time)
      x_t  = (1-t)*x_0 + t*x_1          (linear interpolation)
      target_v = x_1 - x_0              (constant OT velocity)
      loss = ||net(x_t, t) - target_v||^2
    """
    optimizer = optim.Adam(net.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_steps)
    losses = []

    net.train()
    for step in range(1, n_steps + 1):
        # 1. Sample endpoints
        x1 = sample_bimodal(batch_size).to(device)   # (B, 1) — target data
        x0 = torch.randn(batch_size, 1).to(device)   # (B, 1) — noise

        # 2. Sample time uniformly
        t = torch.rand(batch_size).to(device)         # (B,)
        t_col = t.unsqueeze(-1)                        # (B, 1) for broadcasting

        # 3. Interpolate: x_t = (1-t)*x_0 + t*x_1
        xt = (1.0 - t_col) * x0 + t_col * x1         # (B, 1)

        # 4. Target velocity (constant OT direction)
        target_v = x1 - x0                            # (B, 1)

        # 5. Network prediction
        pred_v = net(xt, t)                           # (B, 1)

        # 6. MSE loss
        loss = ((pred_v - target_v) ** 2).mean()

        # 7. Optimise
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        losses.append(loss.item())

        if step % log_every == 0:
            avg = np.mean(losses[-log_every:])
            print(f'Step {step:5d}/{n_steps} | Loss: {avg:.5f} | LR: {scheduler.get_last_lr()[0]:.6f}')

    return losses


net1d = VectorFieldNet1D(hidden_dim=128, n_layers=4).to(device)
print('Starting 1D flow matching training...')
losses_1d = train_flow_matching_1d(net1d, n_steps=5000)
print('Training complete.')

In [ ]:
# ── 4.4  Euler ODE Solver + Sampling ───────────────────────────────────────

@torch.no_grad()
def euler_sample(
    net: nn.Module,
    n_samples: int = 2000,
    n_steps: int = 100,
    data_dim: int = 1,
) -> torch.Tensor:
    """
    Euler method for integrating the learned ODE from t=0 to t=1.

    x_{t+h} = x_t + h * v_theta(x_t, t)

    This is the simplest ODE solver. More accurate solvers (Heun, RK4,
    or adaptive-step solvers from torchdiffeq) can achieve the same quality
    with fewer network evaluations.

    Args:
        net:       trained vector field network
        n_samples: number of samples to generate
        n_steps:   number of Euler integration steps
        data_dim:  dimensionality of the data

    Returns:
        x1: (n_samples, data_dim) — generated samples
    """
    net.eval()
    dt = 1.0 / n_steps

    # Start from pure noise
    x = torch.randn(n_samples, data_dim).to(device)

    # Integrate ODE forward in time from t=0 to t=1
    for step_idx in range(n_steps):
        t_val = step_idx * dt                            # current time
        t_tensor = torch.full((n_samples,), t_val).to(device)
        v = net(x, t_tensor)                             # predicted velocity
        x = x + dt * v                                   # Euler step

    return x.cpu()


# Generate samples with different numbers of steps to show quality vs NFE trade-off
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
step_counts = [5, 20, 50, 100]

for ax, n_steps in zip(axes, step_counts):
    samples = euler_sample(net1d, n_samples=3000, n_steps=n_steps).numpy()
    ax.hist(samples, bins=60, density=True, color='mediumseagreen', alpha=0.7,
            label=f'{n_steps} steps')
    # Overlay true target density
    xs = np.linspace(-5, 5, 500)
    true_density = 0.5 * (1/(0.5*np.sqrt(2*np.pi))) * np.exp(-0.5*((xs+2)/0.5)**2) + \
                   0.5 * (1/(0.5*np.sqrt(2*np.pi))) * np.exp(-0.5*((xs-2)/0.5)**2)
    ax.plot(xs, true_density, 'r--', lw=2, label='True target')
    ax.set_title(f'NFE = {n_steps}')
    ax.set_xlim(-5, 5)
    ax.legend(fontsize=8)
    ax.set_xlabel('x')

plt.suptitle('Generated Samples vs True Target — Effect of Number of Euler Steps', fontsize=12)
plt.tight_layout()
plt.show()

# Training loss curve
fig, ax = plt.subplots(figsize=(8, 3))
window = 50
smoothed = np.convolve(losses_1d, np.ones(window)/window, mode='valid')
ax.plot(smoothed, color='royalblue', lw=1.5, label='Smoothed loss')
ax.set_xlabel('Training step')
ax.set_ylabel('CFM Loss (MSE)')
ax.set_title('1D Flow Matching: Training Convergence')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Final loss (last 100 steps): {np.mean(losses_1d[-100:]):.6f}')

## 5. 2D Toy Example: Gaussian Mixture → Crescent

We now scale to 2D to illustrate how flow matching handles more complex geometry. The source is $\mathcal{N}(0, I_2)$ and the target is a **crescent (half-moon)** distribution — a classic benchmark for generative models because it requires learning a non-trivial curved manifold.

In [ ]:
# ── 5.1  Crescent / Half-Moon Target Distribution ──────────────────────────

def sample_crescent(n: int) -> torch.Tensor:
    """
    Sample from a 2D crescent distribution:
    - Parametrise by angle theta ~ U[0, pi]
    - Radius r ~ N(3, 0.3)
    - Add Gaussian noise in both x and y
    """
    theta = np.random.uniform(0, np.pi, n)
    r = np.random.randn(n) * 0.3 + 3.0
    x = r * np.cos(theta) + np.random.randn(n) * 0.1
    y = r * np.sin(theta) + np.random.randn(n) * 0.1 - 1.0  # shift down
    return torch.tensor(np.stack([x, y], axis=1), dtype=torch.float32)


# Visualise source vs target
source_2d = torch.randn(3000, 2).numpy()
target_2d = sample_crescent(3000).numpy()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(source_2d[:, 0], source_2d[:, 1], s=3, alpha=0.4, c='steelblue')
axes[0].set_title('Source: $\\mathcal{N}(0, I_2)$')
axes[0].set_aspect('equal')
axes[0].set_xlim(-5, 5); axes[0].set_ylim(-5, 5)

axes[1].scatter(target_2d[:, 0], target_2d[:, 1], s=3, alpha=0.4, c='coral')
axes[1].set_title('Target: Crescent Distribution')
axes[1].set_aspect('equal')
axes[1].set_xlim(-5, 5); axes[1].set_ylim(-5, 5)

plt.suptitle('2D Flow Matching: Source and Target', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.2  2D Vector Field Network ───────────────────────────────────────────

class VectorFieldNet2D(nn.Module):
    """
    2D flow matching vector field network.

    Input:  (x_t in R^2, t in R)  — we embed t sinusoidally
    Output: v in R^2              — predicted velocity

    Uses a deeper MLP with layer normalisation for stability in 2D.
    """

    def __init__(self, data_dim: int = 2, hidden_dim: int = 256,
                 n_layers: int = 5, time_embed_dim: int = 64):
        super().__init__()
        self.data_dim = data_dim
        self.time_embed_dim = time_embed_dim

        freqs = torch.exp(
            torch.arange(0, time_embed_dim, 2).float() *
            -(np.log(10000.0) / time_embed_dim)
        )
        self.register_buffer('freqs', freqs)

        self.time_proj = nn.Sequential(
            nn.Linear(time_embed_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

        self.x_proj = nn.Linear(data_dim, hidden_dim)

        layers = []
        for _ in range(n_layers):
            layers.append(nn.Sequential(
                nn.LayerNorm(hidden_dim),
                nn.Linear(hidden_dim, hidden_dim * 2),
                nn.SiLU(),
                nn.Linear(hidden_dim * 2, hidden_dim),
            ))
        self.res_blocks = nn.ModuleList(layers)

        self.out = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, data_dim),
        )

    def time_embedding(self, t: torch.Tensor) -> torch.Tensor:
        t = t.unsqueeze(-1)
        args = t * self.freqs
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        h = self.x_proj(x) + self.time_proj(self.time_embedding(t))
        for block in self.res_blocks:
            h = h + block(h)
        return self.out(h)


# Test
net2d = VectorFieldNet2D().to(device)
dummy_x2 = torch.randn(16, 2).to(device)
dummy_t2 = torch.rand(16).to(device)
v2 = net2d(dummy_x2, dummy_t2)
print(f'2D network output shape: {v2.shape}  (expected: [16, 2])')
print(f'Total parameters: {sum(p.numel() for p in net2d.parameters()):,}')

In [ ]:
# ── 5.3  Train 2D Flow Matching ─────────────────────────────────────────────

def train_flow_matching_2d(
    net: nn.Module,
    n_steps: int = 8000,
    batch_size: int = 1024,
    lr: float = 3e-4,
    log_every: int = 1000,
) -> list:
    """
    CFM training for 2D crescent target.
    Identical algorithm to 1D — only the sampling function changes.
    """
    optimizer = optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_steps)
    losses = []

    net.train()
    for step in range(1, n_steps + 1):
        x1 = sample_crescent(batch_size).to(device)      # (B, 2)
        x0 = torch.randn(batch_size, 2).to(device)        # (B, 2)

        t = torch.rand(batch_size).to(device)              # (B,)
        t_col = t.unsqueeze(-1)                            # (B, 1)

        xt = (1.0 - t_col) * x0 + t_col * x1             # (B, 2)
        target_v = x1 - x0                                 # (B, 2) — OT velocity

        pred_v = net(xt, t)                                # (B, 2)
        loss = ((pred_v - target_v) ** 2).mean()

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        losses.append(loss.item())

        if step % log_every == 0:
            avg = np.mean(losses[-log_every:])
            print(f'Step {step:5d}/{n_steps} | Loss: {avg:.5f}')

    return losses


print('Training 2D flow matching (crescent target)...')
losses_2d = train_flow_matching_2d(net2d, n_steps=8000)
print('Training complete.')

In [ ]:
# ── 5.4  Sample and Visualise 2D Results ────────────────────────────────────

@torch.no_grad()
def euler_sample_2d(
    net: nn.Module,
    n_samples: int = 3000,
    n_steps: int = 100,
    return_trajectory: bool = False,
):
    """
    Euler ODE sampler for 2D flow matching.

    If return_trajectory=True, returns all intermediate positions
    so we can visualise the flow paths.
    """
    net.eval()
    dt = 1.0 / n_steps
    x = torch.randn(n_samples, 2).to(device)

    trajectory = [x.cpu().numpy()] if return_trajectory else None

    for step_idx in range(n_steps):
        t_val = step_idx * dt
        t_tensor = torch.full((n_samples,), t_val).to(device)
        v = net(x, t_tensor)
        x = x + dt * v
        if return_trajectory and (step_idx % 10 == 0):
            trajectory.append(x.cpu().numpy())

    return (x.cpu(), trajectory) if return_trajectory else x.cpu()


# Generate samples
generated_2d, traj = euler_sample_2d(net2d, n_samples=3000, n_steps=100,
                                      return_trajectory=True)
generated_2d = generated_2d.numpy()

# ── Visualisation ───────────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 4))
gs = gridspec.GridSpec(1, 4, figure=fig)

# Panel 1: Source
ax1 = fig.add_subplot(gs[0])
ax1.scatter(source_2d[:1000, 0], source_2d[:1000, 1], s=4, alpha=0.4, c='steelblue')
ax1.set_title('Source ($t=0$)\n$\\mathcal{N}(0,I)$')
ax1.set_xlim(-5, 5); ax1.set_ylim(-5, 5)
ax1.set_aspect('equal')

# Panels 2-3: Intermediate steps
for i, (step_name, step_idx) in enumerate([(r'$t=0.3$', 3), (r'$t=0.7$', 7)]):
    ax = fig.add_subplot(gs[i + 1])
    ax.scatter(traj[step_idx][:1000, 0], traj[step_idx][:1000, 1],
               s=4, alpha=0.4, c='gold')
    ax.set_title(f'Intermediate {step_name}')
    ax.set_xlim(-5, 5); ax.set_ylim(-5, 5)
    ax.set_aspect('equal')

# Panel 4: Generated
ax4 = fig.add_subplot(gs[3])
ax4.scatter(generated_2d[:1000, 0], generated_2d[:1000, 1], s=4, alpha=0.4, c='coral')
ax4.scatter(target_2d[:1000, 0], target_2d[:1000, 1], s=4, alpha=0.2, c='gray',
            label='True target')
ax4.set_title('Generated ($t=1$)\nvs True Target (gray)')
ax4.set_xlim(-5, 5); ax4.set_ylim(-5, 5)
ax4.set_aspect('equal')
ax4.legend(fontsize=8)

plt.suptitle('2D Flow Matching: Particle Trajectories from Noise to Crescent', fontsize=13)
plt.tight_layout()
plt.show()

# Show vector field at a few time points
fig2, axes2 = plt.subplots(1, 3, figsize=(13, 4))
grid_pts = 20
x_lin = np.linspace(-4, 4, grid_pts)
xx, yy = np.meshgrid(x_lin, x_lin)
grid_x = torch.tensor(np.stack([xx.ravel(), yy.ravel()], axis=1),
                       dtype=torch.float32).to(device)

net2d.eval()
with torch.no_grad():
    for ax, t_val in zip(axes2, [0.1, 0.5, 0.9]):
        t_grid = torch.full((grid_x.shape[0],), t_val).to(device)
        vf = net2d(grid_x, t_grid).cpu().numpy()
        ax.quiver(xx, yy,
                  vf[:, 0].reshape(grid_pts, grid_pts),
                  vf[:, 1].reshape(grid_pts, grid_pts),
                  alpha=0.6, color='royalblue', scale=80)
        ax.scatter(target_2d[:500, 0], target_2d[:500, 1],
                   s=3, alpha=0.3, c='coral', label='Target data')
        ax.set_title(f'Vector field at $t={t_val}$')
        ax.set_xlim(-4.5, 4.5); ax.set_ylim(-4.5, 4.5)
        ax.set_aspect('equal')

plt.suptitle('Learned Vector Field at Three Time Points', fontsize=13)
plt.tight_layout()
plt.show()

print('Quality check (compare moments):')
print(f'  Generated mean:  {generated_2d.mean(axis=0).round(2)}')
print(f'  Target mean:     {target_2d.mean(axis=0).round(2)}')
print(f'  Generated std:   {generated_2d.std(axis=0).round(2)}')
print(f'  Target std:      {target_2d.std(axis=0).round(2)}')

## 6. Protein Context: Flow Matching on SE(3) — FrameDiff

### 6.1 Why SE(3)?

A protein backbone is a chain of **rigid frames** — each residue has a local coordinate system defined by a rotation matrix $R \in SO(3)$ and a translation $t \in \mathbb{R}^3$. Together, $(R, t) \in SE(3)$ (the Special Euclidean group in 3D).

We cannot simply apply Euclidean flow matching to rotations because:
- $SO(3)$ is a curved manifold (the sphere of unit quaternions), not a flat vector space
- Linear interpolation between two rotation matrices does not stay on $SO(3)$
- Angles wrap around — the shortest path between two rotations goes through the manifold, not through flat space

### 6.2 FrameDiff: SE(3) Conditional Flow Matching (Yim et al. 2023)

**Key innovations in FrameDiff (arxiv:2302.02277):**

1. **Separate flows for translations and rotations:**
   - Translations $t \in \mathbb{R}^3$: standard Euclidean flow matching
   - Rotations $R \in SO(3)$: geodesic interpolation via the exponential map

2. **SE(3) probability path:** Instead of linear interpolation, use the geodesic path on $SO(3)$:

$$R_t = R_0 \cdot \exp(t \cdot \log(R_0^T R_1))$$

where $\log: SO(3) \to \mathfrak{so}(3)$ is the matrix logarithm (maps to the Lie algebra) and $\exp: \mathfrak{so}(3) \to SO(3)$ is the matrix exponential (maps back to the group).

3. **Tangent space velocity:** The vector field is defined in the **tangent space** of $SO(3)$ at the current point:

$$\dot{R}_t = R_t \cdot \Omega_t, \quad \Omega_t = \frac{d}{dt}\log(R_0^T R_t) \in \mathfrak{so}(3)$$

4. **Network architecture:** An SE(3)-equivariant transformer (IPA — Invariant Point Attention from AlphaFold2) predicts the vector field in the tangent space.

### 6.3 Training Objective

$$\mathcal{L}_{\text{SE(3)-CFM}}(\theta) = \mathbb{E}_{t, (R_1,t_1) \sim p_\text{data}}\left[
  \underbrace{\|v_\theta^t(x_t, t) - u_t^t(x_t | x_1)\|^2}_{\text{translation loss}}
  + \underbrace{\|v_\theta^R(x_t, t) - u_t^R(x_t | x_1)\|_{\mathfrak{so}(3)}^2}_{\text{rotation loss}}
\right]$$

where $\|\cdot\|_{\mathfrak{so}(3)}$ is the Frobenius norm in the Lie algebra.

### 6.4 Why This Matters vs Diffusion

Earlier diffusion-on-SE(3) approaches (e.g. DDPM variants like Framediff-diff) had to carefully design noise schedules on the manifold and estimate scores on curved spaces. Flow matching sidesteps this: you just need a geodesic interpolation and the tangent space velocity — both are computable in closed form for $SO(3)$.

In [ ]:
# ── 6.5  Minimal SE(3) Flow Matching Demonstration ─────────────────────────
#
# We implement the core mathematical operations:
#   - SO(3) exponential and logarithm maps
#   - Geodesic interpolation between two rotations
#   - Tangent space velocity computation
#
# This is NOT a full protein design system (that requires AF2/IPA architecture)
# but illustrates the fundamental SO(3) flow matching mechanics.

def hat(omega: torch.Tensor) -> torch.Tensor:
    """
    Hat map: so(3) vector -> 3x3 skew-symmetric matrix.
    omega: (..., 3)  ->  (..., 3, 3)

    The hat map converts an angular velocity vector [wx, wy, wz] to the
    skew-symmetric matrix form used in SO(3) Lie algebra computations.
    """
    batch = omega.shape[:-1]
    zero = torch.zeros(*batch, device=omega.device, dtype=omega.dtype)
    wx, wy, wz = omega[..., 0], omega[..., 1], omega[..., 2]
    row0 = torch.stack([zero, -wz,  wy], dim=-1)
    row1 = torch.stack([wz,   zero, -wx], dim=-1)
    row2 = torch.stack([-wy,  wx,   zero], dim=-1)
    return torch.stack([row0, row1, row2], dim=-2)  # (..., 3, 3)


def so3_exp(omega: torch.Tensor) -> torch.Tensor:
    """
    Rodrigues' formula: so(3) vector -> SO(3) matrix.
    omega: (..., 3)  ->  R: (..., 3, 3)

    exp(hat(omega)) = I + sin(|omega|)/|omega| * hat(omega)
                        + (1-cos(|omega|))/|omega|^2 * hat(omega)^2
    """
    theta = omega.norm(dim=-1, keepdim=True).unsqueeze(-1)  # (..., 1, 1)
    theta = theta.clamp(min=1e-8)
    K = hat(omega)   # (..., 3, 3)
    I = torch.eye(3, device=omega.device, dtype=omega.dtype).expand_as(K)
    R = I + (torch.sin(theta) / theta) * K + \
        ((1 - torch.cos(theta)) / theta**2) * (K @ K)
    return R  # (..., 3, 3)


def so3_log(R: torch.Tensor) -> torch.Tensor:
    """
    Matrix logarithm: SO(3) matrix -> so(3) vector.
    R: (..., 3, 3)  ->  omega: (..., 3)

    Uses the formula: theta = arccos((trace(R)-1)/2)
                      omega = theta/(2*sin(theta)) * vee(R - R^T)
    where vee is the inverse of hat.
    """
    # Rotation angle
    trace = R[..., 0, 0] + R[..., 1, 1] + R[..., 2, 2]  # (...,)
    cos_theta = ((trace - 1.0) / 2.0).clamp(-1.0 + 1e-7, 1.0 - 1e-7)
    theta = torch.acos(cos_theta)  # (...,)

    # Skew-symmetric part
    R_minus_RT = R - R.transpose(-1, -2)  # (..., 3, 3)

    # vee map: extract [R32-R23, R13-R31, R21-R12] / 2
    omega_unnorm = torch.stack([
        R_minus_RT[..., 2, 1],
        R_minus_RT[..., 0, 2],
        R_minus_RT[..., 1, 0],
    ], dim=-1)  # (..., 3)

    # Scale by theta / (2 * sin(theta))
    scale = theta / (2.0 * torch.sin(theta).clamp(min=1e-8))  # (...,)
    return scale.unsqueeze(-1) * omega_unnorm  # (..., 3)


def so3_geodesic_interp(R0: torch.Tensor, R1: torch.Tensor,
                         t: float) -> torch.Tensor:
    """
    Geodesic interpolation on SO(3) at time t in [0, 1].
    R_t = R0 * exp(t * log(R0^T * R1))

    This is the SE(3) analogue of linear interpolation.
    At t=0 returns R0; at t=1 returns R1.
    """
    dR = R0.transpose(-1, -2) @ R1          # R0^T R1 — relative rotation
    log_dR = so3_log(dR)                     # tangent vector (so(3))
    interp_log = t * log_dR                  # scale tangent vector by t
    Rt = R0 @ so3_exp(interp_log)            # push back to SO(3)
    return Rt


def so3_velocity(R0: torch.Tensor, R1: torch.Tensor) -> torch.Tensor:
    """
    Constant tangent space velocity for the geodesic path from R0 to R1.
    Returns omega in so(3): the angular velocity vector.
    (Analogous to x1 - x0 in Euclidean flow matching.)
    """
    dR = R0.transpose(-1, -2) @ R1
    return so3_log(dR)  # (..., 3)


# ── Demonstration ───────────────────────────────────────────────────────────
torch.manual_seed(0)
B = 4  # batch of 4 residue frames

# Random source and target rotations
omega_source = torch.randn(B, 3) * 0.5  # small random rotations
omega_target = torch.randn(B, 3) * 1.0
R0 = so3_exp(omega_source)  # (B, 3, 3)
R1 = so3_exp(omega_target)  # (B, 3, 3)

print('SO(3) Geodesic Interpolation Demo')
print('===================================')

# Interpolate at t=0, 0.25, 0.5, 0.75, 1.0 and check they stay on SO(3)
for t_val in [0.0, 0.25, 0.5, 0.75, 1.0]:
    Rt = so3_geodesic_interp(R0, R1, t_val)
    # Check: R^T R = I  and  det(R) = 1
    RtT_Rt = Rt.transpose(-1, -2) @ Rt
    I_approx = RtT_Rt - torch.eye(3).unsqueeze(0)
    det = torch.det(Rt)
    print(f't={t_val:.2f} | max |R^TR - I|: {I_approx.abs().max():.2e} | '
          f'det(R): {det.mean():.6f} (should be ~1.0)')

print()
# Compute the OT velocity (constant along geodesic)
omega = so3_velocity(R0, R1)
print(f'Tangent velocity omega shape: {omega.shape}')
print(f'Rotation angle to travel: {omega.norm(dim=-1).numpy().round(3)} radians')
print()
print('Key insight: the tangent velocity omega = log(R0^T R1) is CONSTANT along')
print('the geodesic path, exactly like x1-x0 is constant in Euclidean CFM.')

## 7. Boltz-1 Connection: Replacing DDPM with Flow Matching

### 7.1 AlphaFold3 and the Diffusion Head

AlphaFold3 (Abramson et al. 2024, Nature doi:10.1038/s41586-024-07487-w) introduced a **diffusion-based structure module** to replace the equivariant structure module of AF2. The key change:

- AF2: deterministic structure module, output directly from attention
- AF3: diffusion generative model, conditioned on trunk representations

AF3 uses a **VP-SDE** (variance-preserving stochastic differential equation) for its diffusion head, which is essentially DDPM in continuous time.

### 7.2 Boltz-1 Architecture Change

Boltz-1 (MIT, 2024) is an open-source reimplementation and improvement of AF3. One of its most impactful changes: **replacing the DDPM head with flow matching**.

| Component | AlphaFold3 | Boltz-1 |
|-----------|-----------|--------|
| Trunk | Pairformer (identical) | Pairformer |
| Structure module type | VP-SDE diffusion | Flow matching (OT-CFM) |
| Noise schedule | Cosine/linear VP schedule | None (linear interpolation) |
| Sampling steps needed | ~200 (DDPM) or ~50 (DDIM) | ~20-50 (Euler/Heun) |
| Training target | Score / $\epsilon$-prediction | Velocity (x1 - x0) |
| Key benefit | Well-studied stability | Faster inference, cleaner paths |

### 7.3 Why Flow Matching Helps for Structure Prediction

**1. Straighter paths = fewer steps:** Protein structures are not that far from Gaussian noise in feature space (the pairformer already does most of the heavy lifting). The diffusion head just needs to "sharpen" the structure. With straight OT paths, 20 steps can be enough.

**2. No noise schedule tuning:** DDPM requires careful tuning of the $\beta_t$ schedule. Get it wrong and the model either over-smooths early or misses fine details. Flow matching with linear interpolation has no such hyperparameter.

**3. Better conditioning:** When the model is conditioned on the Pairformer trunk (which already encodes a lot of structural information), the flow becomes even more direct — the vector field mostly points from noise to the conditioned structure, not an arbitrary distribution.

**4. Conditional generation is natural:** For tasks like binding site design or partial structure completion, the conditional CFM objective naturally handles partial evidence: just fix the known coordinates and only flow the unknown ones.

### 7.4 Practical Training Detail

In Boltz-1 (and generalised flow matching for biomolecules), the full training loss is:

$$\mathcal{L} = \lambda_{\text{CFM}} \cdot \mathcal{L}_{\text{CFM}} + \lambda_{\text{FAPE}} \cdot \mathcal{L}_{\text{FAPE}} + \lambda_{\text{bond}} \cdot \mathcal{L}_{\text{bond}} + \lambda_{\text{conf}} \cdot \mathcal{L}_{\text{conf}}$$

The CFM loss provides the generative signal; FAPE (Frame-Aligned Point Error) ensures physical correctness of the predicted frames; bond and confidence losses add structural constraints.

## 8. Interview Preparation

### Q1: What is the key difference between flow matching and DDPM training objectives?

**Answer:**

DDPM trains a network to predict the **noise** $\epsilon$ that was added to corrupt a data point at step $t$, using the objective $\mathbb{E}[\|\epsilon_\theta(x_t, t) - \epsilon\|^2]$. The training signal requires a specific noise schedule (the $\beta_t$ sequence) to define what $x_t$ looks like.

Flow matching trains a network to predict the **velocity** (direction of movement) from source to target, using the conditional flow matching objective $\mathbb{E}[\|v_\theta(x_t, t) - (x_1 - x_0)\|^2]$. The training signal is purely geometric: for the optimal transport path, the target velocity is just the constant displacement $x_1 - x_0$, and $x_t = (1-t)x_0 + tx_1$ by construction. No noise schedule is needed.

The practical consequence: flow matching paths are straighter, so ODE integration at inference needs far fewer steps (typically 10-50 vs 100-1000 for DDPM).

---

### Q2: What is an ODE solver in this context, and why does the choice matter?

**Answer:**

At inference time, flow matching requires integrating the ODE $dx/dt = v_\theta(x_t, t)$ from $t=0$ to $t=1$ to generate samples. An ODE solver approximates this integral numerically by taking discrete steps.

- **Euler method:** $x_{t+h} = x_t + h \cdot v_\theta(x_t, t)$ — 1 network evaluation per step, first-order accuracy. Simple but requires many steps for accuracy.
- **Heun method (RK2):** Takes a predictor step (Euler), then corrects — 2 evaluations per step, second-order accuracy. Often 2x fewer steps needed vs Euler for the same quality.
- **Runge-Kutta 4 (RK4):** 4 evaluations per step, fourth-order accuracy. Best for high precision but expensive.
- **Adaptive solvers (Dopri5, etc.):** Adjust step size automatically based on local error — can be very efficient when the vector field is smooth (which it is for flow matching with straight paths).

The choice matters because each network evaluation is the bottleneck (the neural network forward pass is expensive). Fewer steps = faster inference. This is why flow matching is preferred over DDPM for deployment: the straighter paths are better-conditioned for high-order ODE solvers.

---

### Q3: Why is flow matching on SE(3) non-trivial, and how does FrameDiff handle it?

**Answer:**

Protein backbone frames live on the SE(3) manifold (rotations + translations). SE(3) is curved — specifically, the rotation component SO(3) is a sphere-like manifold. You cannot interpolate between two rotation matrices by linear combination in matrix space because the result will not be a valid rotation matrix (it won't satisfy $R^T R = I$, $\det(R) = 1$).

FrameDiff (Yim et al. 2023) handles this by:
1. **Decomposing** SE(3) into translations ($\mathbb{R}^3$, Euclidean flow matching applies directly) and rotations ($SO(3)$, needs manifold-aware treatment).
2. **Geodesic interpolation** for rotations: $R_t = R_0 \exp(t \log(R_0^T R_1))$ using the matrix exponential and logarithm maps, which stay on $SO(3)$.
3. **Tangent space velocity:** The network predicts the velocity as a vector in the tangent space of $SO(3)$ at the current point (a skew-symmetric matrix / axis-angle vector), not as a rotation matrix directly. This ensures the velocity is in the right space for ODE integration.

The result is a clean CFM objective split into separate Euclidean and SO(3) components, both tractable.

---

### Q4: What is the continuity equation and why is it central to flow matching?

**Answer:**

The continuity equation is the fundamental PDE linking the probability density $p_t(x)$ to the vector field $u_t(x)$ that generates it:

$$\frac{\partial p_t}{\partial t} + \nabla \cdot (p_t u_t) = 0$$

This says: the rate of change of probability density at a point equals the negative divergence of the probability flux ($p_t u_t$). In fluid mechanics, it's the statement of mass conservation for an incompressible fluid.

In flow matching, this equation tells us:
- **Given** a target probability path $p_t$ (e.g. the linear interpolation between $p_0$ and $p_1$), **there exists** a vector field $u_t$ that generates it (the marginal vector field)
- The CFM objective implicitly minimises the difference between the learned and true vector fields by matching conditional velocities, which (by the CFM theorem) is equivalent to matching the marginal vector field that satisfies the continuity equation

This is why the CFM loss is sound: minimising it implies the learned ODE generates the correct probability path.

---

### Q5: Why does Boltz-1 prefer flow matching over DDPM for structure prediction, and what are the trade-offs?

**Answer:**

**Advantages of flow matching for structure prediction:**
1. **Faster inference:** Straighter paths → fewer ODE steps needed (20 vs 200+). Critical for structure prediction where you might need thousands of samples.
2. **No noise schedule tuning:** Eliminates a major hyperparameter. The cosine/linear $\beta_t$ schedule in DDPM requires careful tuning and can fail on out-of-distribution inputs.
3. **Exact likelihood:** CNFs (continuous normalizing flows) provide exact log-likelihood via the instantaneous change-of-variables formula $d\log p_t = -\nabla \cdot u_t \, dt$. DDPM only provides a lower bound (ELBO).
4. **Simpler conditioning:** For conditional generation (e.g. binding interface design), the CFM objective naturally handles conditioning without requiring classifier-free guidance scale tuning.

**Trade-offs:**
1. **Less studied:** DDPM has a larger literature, more ablations, and better-understood failure modes.
2. **ODE solvers needed:** Require choosing/implementing a numerical ODE solver; diffusion models can use closed-form DDIM.
3. **Deterministic paths:** Flow matching ODE is deterministic; adding stochasticity (like in DDPM with the $\sigma_t$ term) can sometimes help escape local minima. Flow matching can add stochasticity via SDE formulations but it's not standard.
4. **Mode coverage:** DDPM's stochasticity can help cover multimodal distributions (multiple conformations); pure ODE flow matching may collapse to fewer modes.

## 9. Debug Exercise: Broken Flow Matching Training Loop

The code cell below contains **three deliberate bugs**. Find and fix them. Each bug will cause either silent failure (training appears to run but generates garbage) or a runtime error.

**Hint:** Read the comments carefully. Think about:
1. The time direction: which end is noise, which is data?
2. Normalisation: what does the loss function actually receive?
3. The ODE integration direction: which way should time flow during sampling?

In [ ]:
# ── DEBUG EXERCISE: Find and fix the 3 bugs ─────────────────────────────────
#
# This is a minimal flow matching implementation with 3 deliberate errors.
# Run the code, observe the symptoms, then fix all three bugs.

class BuggyVectorFieldNet(nn.Module):
    """Simple MLP for debugging — no sinusoidal embedding."""
    def __init__(self):
        super().__init__()
        # Concatenates [x_t (2D), t (1D)] -> 3 input features
        self.net = nn.Sequential(
            nn.Linear(3, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 2),
        )

    def forward(self, x, t):
        # x: (B, 2), t: (B,)
        t_col = t.unsqueeze(-1)  # (B, 1)
        inp = torch.cat([x, t_col], dim=-1)  # (B, 3)
        return self.net(inp)


def buggy_train_step(net, x1_batch, optimizer):
    """
    One training step of CFM.
    x1_batch: (B, 2) real data samples
    """
    B = x1_batch.shape[0]
    x0 = torch.randn(B, 2)  # source noise

    t = torch.rand(B)
    t_col = t.unsqueeze(-1)

    # BUG 1: Wrong interpolation direction
    # This goes from x1 (data) to x0 (noise) instead of x0 -> x1
    xt = (1.0 - t_col) * x1_batch + t_col * x0   # <- BUG: x1 and x0 are swapped

    # BUG 2: Wrong target velocity
    # The target velocity for OT-CFM should be (x1 - x0), not (x0 - x1)
    target_v = x0 - x1_batch  # <- BUG: sign is flipped

    pred_v = net(xt, t)

    # BUG 3: Missing normalisation — dividing by ||pred_v|| destroys the signal
    # The loss should be ||pred_v - target_v||^2, not ||pred_v/||pred_v|| - target_v||^2
    pred_v_normed = pred_v / (pred_v.norm(dim=-1, keepdim=True) + 1e-8)  # <- BUG
    loss = ((pred_v_normed - target_v) ** 2).mean()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()


@torch.no_grad()
def buggy_sample(net, n_samples=500, n_steps=50):
    """
    ODE sampler — also buggy.
    """
    net.eval()
    dt = 1.0 / n_steps
    x = torch.randn(n_samples, 2)   # start from noise

    for step_idx in range(n_steps):
        t_val = step_idx * dt
        t_tensor = torch.full((n_samples,), t_val)
        v = net(x, t_tensor)
        # The Euler step SUBTRACTS instead of ADDS the velocity
        # This integrates BACKWARD in time (from t=1 to t=0, i.e. data -> noise)
        # Fix: change minus to plus
        x = x - dt * v   # <- NOTE: this is in the sampler, bugs 1-3 are in train

    return x


print('BUGGY CODE — Symptoms to observe:')
print('  1. Training loss will NOT converge or will converge to wrong value')
print('  2. Generated samples will NOT match the target distribution')
print('  3. The vector field will point in the wrong direction')
print()
print('=== BUG LOCATIONS ===')
print('Bug 1 (line ~xt=...):     Interpolation swaps x0 and x1 — trains wrong direction')
print('Bug 2 (line ~target_v=):  Target velocity has wrong sign — points noise->data backwards')
print('Bug 3 (line ~pred_v_normed=): Normalising velocity destroys magnitude information')
print()
print('=== FIXES ===')
print('Bug 1: xt = (1.0 - t_col) * x0 + t_col * x1_batch')
print('Bug 2: target_v = x1_batch - x0')
print('Bug 3: loss = ((pred_v - target_v) ** 2).mean()  [remove normalisation]')
print()
print('Note: buggy_sample also has x = x - dt*v which should be x = x + dt*v')
print('This is a 4th bonus bug in the sampler (independent of training bugs).')

In [ ]:
# ── FIXED VERSION — verify the fixes work ───────────────────────────────────

def fixed_train_step(net, x1_batch, optimizer):
    """Corrected CFM training step with all 3 bugs fixed."""
    B = x1_batch.shape[0]
    x0 = torch.randn(B, 2)

    t = torch.rand(B)
    t_col = t.unsqueeze(-1)

    # FIX 1: Correct interpolation — x0 (noise) at t=0, x1 (data) at t=1
    xt = (1.0 - t_col) * x0 + t_col * x1_batch

    # FIX 2: Correct OT velocity — points from source to target
    target_v = x1_batch - x0

    pred_v = net(xt, t)

    # FIX 3: No normalisation — use raw MSE on velocity vectors
    loss = ((pred_v - target_v) ** 2).mean()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()


@torch.no_grad()
def fixed_sample(net, n_samples=500, n_steps=50):
    """Corrected sampler: integrate forward in time (noise -> data)."""
    net.eval()
    dt = 1.0 / n_steps
    x = torch.randn(n_samples, 2)
    for step_idx in range(n_steps):
        t_val = step_idx * dt
        t_tensor = torch.full((n_samples,), t_val)
        v = net(x, t_tensor)
        x = x + dt * v  # FIX: + not -
    return x


# Quick training run to verify fixes
fix_net = BuggyVectorFieldNet()
fix_opt = optim.Adam(fix_net.parameters(), lr=1e-3)

crescent_all = sample_crescent(10000)
fix_losses = []
for step in range(2000):
    idx = torch.randperm(len(crescent_all))[:256]
    x1_b = crescent_all[idx]
    fix_losses.append(fixed_train_step(fix_net, x1_b, fix_opt))

print(f'Fixed training — initial loss: {np.mean(fix_losses[:50]):.4f}')
print(f'Fixed training — final loss:   {np.mean(fix_losses[-50:]):.4f}')

# Sample and visualise
gen_fixed = fixed_sample(fix_net, n_samples=1000, n_steps=100).numpy()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(gen_fixed[:, 0], gen_fixed[:, 1], s=5, alpha=0.4, c='mediumseagreen')
axes[0].set_title('Fixed model — generated')
axes[0].set_xlim(-5, 5); axes[0].set_ylim(-4, 4)
axes[0].set_aspect('equal')

axes[1].scatter(target_2d[:1000, 0], target_2d[:1000, 1], s=5, alpha=0.4, c='coral')
axes[1].set_title('True crescent target')
axes[1].set_xlim(-5, 5); axes[1].set_ylim(-4, 4)
axes[1].set_aspect('equal')

plt.suptitle('Debug Exercise: Fixed Model vs Target', fontsize=13)
plt.tight_layout()
plt.show()

## 10. Mastery Check (Bloom's Taxonomy)

Work through these questions in order — they progress from recall to synthesis.

---

### Level 1 — Remember

**Q1:** Write out the conditional flow matching (CFM) training objective $\mathcal{L}_{\text{CFM}}(\theta)$ and identify each term. What is the target velocity for the optimal transport path?

<details><summary>Answer</summary>

$$\mathcal{L}_{\text{CFM}}(\theta) = \mathbb{E}_{t \sim U[0,1],\, x_1 \sim p_{\text{data}},\, x_0 \sim p_0} \left[ \|v_\theta(x_t, t) - (x_1 - x_0)\|^2 \right]$$

- $v_\theta(x_t, t)$: the network's predicted velocity at interpolated point $x_t$ at time $t$
- $x_1 - x_0$: the target OT velocity (constant along each path)
- $x_t = (1-t)x_0 + tx_1$: the linearly interpolated point

</details>

---

### Level 2 — Understand

**Q2:** Explain in one paragraph why the conditional flow matching loss has the same gradient as the (intractable) marginal flow matching loss. What mathematical property makes this work?

<details><summary>Answer</summary>

The key is that the CFM and FM objectives differ only by a term that does not depend on $\theta$. Specifically:

$$\mathcal{L}_{\text{CFM}}(\theta) = \mathcal{L}_{\text{FM}}(\theta) + C(\text{data})$$

where $C$ is a constant with respect to the network parameters. This follows because expanding the squared norm in $\mathcal{L}_{\text{CFM}}$ and marginalising over $x_1$ recovers $\mathcal{L}_{\text{FM}}$ plus cross-terms that cancel due to the tower property of conditional expectations. Therefore $\nabla_\theta \mathcal{L}_{\text{CFM}} = \nabla_\theta \mathcal{L}_{\text{FM}}$ and gradient descent on either objective leads to the same solution.

</details>

---

### Level 3 — Apply

**Q3:** You train a flow matching model on 2D data and at inference time generate samples with 5 Euler steps. The samples look blurry and do not match the target distribution. What are three things you should try to improve quality, and which would you try first?

<details><summary>Answer</summary>

1. **Increase the number of steps** (easiest): 5 is very few. Try 50-100 Euler steps first. The error in Euler integration is $O(h^2)$ per step and $O(h)$ globally, so doubling steps halves the error.

2. **Switch to a higher-order solver**: Replace Euler with the Heun method (2nd order, same NFE ×2 but much more accurate) or RK4. This should be tried before adding more Euler steps since it's more principled.

3. **Train longer or increase model capacity**: If the vector field is poorly learned (training loss still high), more steps won't help much. Check the training loss curve.

Try in order: (1) more steps → (2) better solver → (3) better model.

</details>

---

### Level 4 — Analyse

**Q4:** A colleague claims that diffusion models and flow matching are "completely different" methods. Write a 3-4 sentence technical argument explaining why they are more similar than they appear, referencing at least one mathematical connection.

<details><summary>Answer</summary>

Diffusion models can be expressed as stochastic differential equations (SDEs) of the form $dx = f(x,t)dt + g(t)dW$ where $W$ is Brownian motion. Every SDE has a corresponding probability flow ODE (Song et al. 2021) with the same marginal distributions: $dx = [f(x,t) - \frac{1}{2}g(t)^2 \nabla_x \log p_t(x)] dt$. Flow matching with straight OT paths corresponds exactly to such a probability flow ODE, where the drift equals the score function plus a correction term. The practical differences (training objective, number of steps, noise schedule) are engineering choices layered on top of this shared mathematical foundation; in the limit of infinitesimally small steps, both methods integrate the same marginal probability path.

</details>

---

### Level 5 — Evaluate / Synthesise

**Q5:** You are designing a new protein backbone generation model and must choose between (a) VP-SDE diffusion (like AF3) and (b) OT-CFM flow matching (like Boltz-1). Your constraints: inference speed is critical (clinical setting, must generate 10,000 structures per hour on a single A100), but training stability matters too (limited compute budget). Which do you choose, and what specific architectural choices do you make for the structure generation head?

<details><summary>Answer</summary>

Choose **OT-CFM flow matching** for this setting. Justification:

**Inference speed (critical):** At 10k structures/hour on A100, you need ~0.36 seconds per structure. With VP-SDE you need 50+ steps × forward pass time; flow matching can achieve comparable quality in 10-20 steps with a Heun solver. This gives a ~3-5x speedup in inference, which is decisive.

**Training stability:** Flow matching is actually *more* stable than DDPM for limited compute because (a) the loss is simpler (no noise schedule annealing), (b) gradients are more consistent across time steps (the target velocity $x_1 - x_0$ has similar magnitude for all $t$, while DDPM gradients vary dramatically with $\beta_t$).

**Architectural choices for the structure head:**
- Separate OT-CFM flows for translations ($\mathbb{R}^3$) and rotations ($SO(3)$ via geodesic interpolation), as in FrameDiff
- IPA (Invariant Point Attention) as the vector field network backbone, conditioned on Pairformer trunk representations
- Heun integrator at inference (2nd order, 2 NFE per step, 10 steps total = 20 NFE)
- Optional: minibatch OT pairing (match source samples to nearest data points) to make paths even straighter and reduce NFE further

</details>

## 11. Cross-References

| Topic | Where to find it | Connection to this notebook |
|-------|-----------------|----------------------------|
| **DDPM — the diffusion model this replaces** | Module 12/01 (`01_diffusion_protein_design.ipynb`) | Section 3 (side-by-side comparison) builds directly on 12/01's DDPM implementation |
| **Pairformer trunk (AF3 architecture)** | Module 07/01 (`07_alphafold3_core/01_alphafold3_architecture.ipynb`) | Section 7 (Boltz-1) assumes familiarity with the Pairformer; the flow matching head replaces AF3's diffusion head |
| **FAPE loss and rigid frames** | Module 07/02 (`07_alphafold3_core/02_training_loop.ipynb`) | SE(3) frames in Section 6 are the same frames used in FAPE |
| **IPA (Invariant Point Attention)** | Module 07/01 (AF3 architecture) | The vector field network on SE(3) in FrameDiff uses IPA; understanding IPA is prerequisite |
| **GNNs and equivariance** | Module 06/02 (`06_structural_ml_gnns/`) | The SE(3)-equivariance requirement for the FrameDiff vector field network is the same equivariance discussed in module 06 |
| **Self-supervised pre-training** | Module 15 (`15_self_supervised_learning/`) | Foundation models pre-trained with MLM/SimCLR can initialise the flow matching trunk, reducing training data needed |
| **Bayesian uncertainty** | Module 13 (`13_bayesian_methods/`) | Flow matching generates diverse samples; Bayesian methods can post-process these for calibrated confidence |
| **LoRA fine-tuning** | Module 10/01 (`10_openfold3_finetuning/`) | Flow matching models (like FrameDiff) can be fine-tuned for specific protein families using LoRA on the IPA layers |

## 12. Resources and Citations

### Primary Papers

1. **Lipman, Y., Chen, R. T. Q., Ben-Hamu, H., Nickel, M., & Le, M. (2022).**  
   *Flow Matching for Generative Modeling.*  
   arXiv:2210.02747. https://arxiv.org/abs/2210.02747  
   **The foundational paper.** Introduces the CFM objective and proves its equivalence to FM. Read the proof of Theorem 1 carefully.

2. **Yim, J., Trippe, B. L., De Bortoli, V., Mathieu, E., Doucet, A., Barzilay, R., & Jaakkola, T. (2023).**  
   *SE(3) diffusion model with application to protein backbone generation.*  
   arXiv:2302.02277. https://arxiv.org/abs/2302.02277  
   **FrameDiff.** The paper that lifted flow matching to SE(3) for protein backbones. Read Section 3 for the SO(3) probability path derivation.

3. **Abramson, J., Adler, J., Dunger, J., et al. (2024).**  
   *Accurate structure prediction of biomolecular interactions with AlphaFold 3.*  
   *Nature*, 630, 493–500. doi:10.1038/s41586-024-07487-w  
   **AlphaFold3.** Sections 3-4 describe the diffusion head that Boltz-1 later replaced with flow matching.

4. **Albergo, M. S., Vanden-Eijnden, E. (2023).**  
   *Building Normalizing Flows with Stochastic Interpolants.*  
   arXiv:2209.15571. https://arxiv.org/abs/2209.15571  
   **Stochastic interpolants.** Concurrent and complementary to Lipman et al.; introduces the stochastic interpolant framework that unifies diffusion and flow matching.

5. **Liu, X., Gong, C., & Liu, Q. (2023).**  
   *Flow Straight and Fast: Learning to Generate and Transfer Data with Rectified Flow.*  
   arXiv:2209.03003. https://arxiv.org/abs/2209.03003  
   **Rectified Flow.** Alternative (and independently discovered) formulation of flow matching with straight paths. Used in Stable Diffusion 3.

6. **Song, Y., Sohl-Dickstein, J., Kingma, D. P., Kumar, A., Ermon, S., & Poole, B. (2021).**  
   *Score-Based Generative Modeling through Stochastic Differential Equations.*  
   arXiv:2011.13456. https://arxiv.org/abs/2011.13456  
   **Score-based diffusion.** Essential background — establishes the ODE-SDE duality and the probability flow ODE that connects diffusion to flow matching.

### Blog Posts and Tutorials

- **Lilian Weng — Flow Matching:** https://lilianweng.github.io/posts/2023-12-22-flow-matching/  
  Best written introduction to the topic. Read before the Lipman paper.

- **Will Whitney — An Introduction to Flow Matching:** https://mlg.eng.cam.ac.uk/blog/2024/01/20/flow-matching.html  
  Cambridge ML Group blog, very clear mathematical exposition.

- **Boltz-1 Technical Report:** https://github.com/jwohlwend/boltz  
  The open-source codebase for Boltz-1. Look at `boltz/model/heads/diffusion.py` for the flow matching head implementation.

### Key Implementation Libraries

- **torchdiffeq:** https://github.com/rtqichen/torchdiffeq — Production ODE solvers (Dormand-Prince, Adams) for PyTorch. Use `odeint` for adaptive step-size integration.
- **torchsde:** https://github.com/google-research/torchsde — For SDE-based models (DDPM, score-based).
- **POT (Python Optimal Transport):** https://pythonot.github.io/ — For computing optimal transport plans and implementing minibatch OT pairing.

In [ ]:
# ── Final Summary: What We Built ────────────────────────────────────────────

print('=' * 60)
print('NOTEBOOK COMPLETE: Flow Matching for Protein Design')
print('=' * 60)
print()
print('IMPLEMENTED:')
print('  1. VectorFieldNet1D — sinusoidal time embedding + residual MLP')
print('  2. CFM training loop — OT path, constant velocity target')
print('  3. Euler ODE sampler — controllable NFE trade-off')
print('  4. VectorFieldNet2D — LayerNorm + deep MLP for 2D case')
print('  5. 2D Gaussian -> crescent flow (trained and visualised)')
print('  6. SO(3) exp, log, geodesic interpolation (hat map, Rodrigues)')
print('  7. Debug exercise (3 bugs + fixed version)')
print()
print('KEY MATHEMATICAL RESULTS:')
print('  - Continuity equation links p_t and u_t')
print('  - CFM theorem: L_CFM and L_FM have equal gradients')
print('  - OT velocity = x1 - x0 (constant, independent of t)')
print('  - SO(3) geodesic: R_t = R0 * exp(t * log(R0^T R1))')
print()
print('CONNECTIONS MADE:')
print('  - Flow matching vs DDPM (Section 3)')
print('  - FrameDiff SE(3) formulation (Section 6)')
print('  - Boltz-1 replacing AF3 diffusion head (Section 7)')
print()
print('NEXT STEPS:')
print('  - Module 12/01: Review DDPM to solidify the comparison')
print('  - Module 07/01: Understand Pairformer trunk for AF3/Boltz-1 context')
print('  - Read: Lipman et al. 2022 (arxiv:2210.02747)')
print('  - Code: Explore Boltz-1 source at github.com/jwohlwend/boltz')

## Notebook Complete

**You can now:**
- Implement flow matching as a continuous normalising flow for protein backbone generation
- Compare flow matching with diffusion models and explain the key differences in training and sampling

**Where this fits:**
- Previous: 01_diffusion_protein_design
- Next: 13_bayesian_methods/01_bayesian_ml_uncertainty — Module 13 — check the README

**Before moving on, check:**
- [ ] All code cells ran without errors
- [ ] You understand what each function does (read the docstrings)
- [ ] You can explain the key concept in one sentence without looking at notes